In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalizability Evaluation for Relations Circuit Analysis

This notebook evaluates whether the findings in the repository generalize beyond the original experimental setting.

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data
- **GT3**: Method / Specificity Generalizability

## Repository Location
`/net/scratch2/smallyan/relations_eval`

In [2]:
# First, let's explore the repository structure to understand the research
import os

repo_path = '/net/scratch2/smallyan/relations_eval'
print("Repository structure:")
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit files shown per directory
        print(f'{subindent}{file}')
    if len(files) > 10:
        print(f'{subindent}... and {len(files) - 10} more files')

Repository structure:
relations_eval/
  pyproject.toml
  experiments.py
  CodeWalkthrough.md
  plan.md
  requirements.txt
  schematic-wide.png
  invoke.yaml
  LICENSE
  .gitignore
  tasks.py
  ... and 1 more files
  evaluation/
    self_matching.ipynb
    consistency_evaluation.json
    replications/
      documentation_replication.md
      evaluation_replication.md
      self_replication_evaluation.json
      replication.ipynb
    replication_eval/
      documentation_eval_summary.json
      documentation_evaluation_summary.md
  hparams/
    gptj/
      occupation_age.json
      task_done_by_tool.json
      star_constellation_name.json
      plays_pro_sport.json
      country_currency.json
      superhero_archnemesis.json
      superhero_person.json
      name_religion.json
      city_in_country.json
      univ_degree_gender.json
      ... and 37 more files
    gpt2-xl/
      food_from_country.json
      substance_phase_of_matter.json
      superhero_archnemesis.json
      fruit_insid

      figs/
        varying_n.pdf
        varying_n-crop.pdf
        efficacy_lre_models.pdf
        vaying_n_legend.pdf
        faithfulness_lre_models.pdf
        llama-13b/
          legend_faithfulness_prompting.pdf
          llama-13b-faithfulness_baselines.pdf
          llama-13b-faithfulness_baselines_prompting.pdf
      stats/
        data.json
        range_stats.json
  src/
    data.py
    operators.py
    __init__.py
    models.py
    corner.py
    editors.py
    metrics.py
    functional.py
    hparams.py
    sweeps.py
    ... and 2 more files
    __pycache__/
      __init__.cpython-311.pyc
      functional.cpython-311.pyc
      models.cpython-311.pyc
      lens.cpython-311.pyc
      editors.cpython-311.pyc
      operators.cpython-311.pyc
      data.cpython-311.pyc
    utils/
      experiment_utils.py
      logging_utils.py
      __init__.py
      typing.py
      env_utils.py
      dataclasses_utils.py
      tokenizer_utils.py
      viz_utils.py
      sweep_utils.py
      _

      pack/
        pack-feff4f7b78314b50aa2699774aa5b5f1d9df9f85.idx
        pack-feff4f7b78314b50aa2699774aa5b5f1d9df9f85.rev
        pack-feff4f7b78314b50aa2699774aa5b5f1d9df9f85.pack
      b3/
        22116ea72b1e62c5d841f77c0e060ae90f93a7
      f1/
        fc88a265469597238e60e10ead955ac6635c0e
      info/
      d5/
        323c6e5792201f929d0f225d5acf2d750476d6
    logs/
      HEAD
      refs/
        remotes/
          origin/
            HEAD
            main
        heads/
          main
  demo/
    attribute_lens.ipynb
    demo.ipynb
    attribute_lens.png
    causality-crop.png
  scripts/
    sweep_n_icl.py
    knowns.py
    dataset_stats.py
    sweep_prompt.py
    __init__.py
    evaluate.py
    sweep.py
    sweep_full_rank.py
    caching/
      cache_lres.py
      interpolation.py
      explain_beta.py
    baselines/
      translation_baseline.py
      faithfulness_baselines.py
      efficacy_baselines.py


In [3]:
# Read the plan and documentation to understand what this research is about
with open(os.path.join(repo_path, 'plan.md'), 'r') as f:
    plan_content = f.read()
print("=== PLAN.MD ===")
print(plan_content[:5000])

=== PLAN.MD ===
# Plan
## Objective
Investigate how transformer language models represent and decode relational knowledge, specifically testing whether relation decoding can be well-approximated by linear transformations on subject representations.

## Hypothesis
1. For a variety of relations, transformer LMs decode relational knowledge directly from subject entity representations at intermediate layers.
2. For each relation, the decoding procedure is approximately affine (linear relational embedding), expressed as LRE(s) = Wrs + br mapping subject s to object o.
3. These affine transformations can be computed directly from the LM Jacobian on a prompt expressing the relation (∂o/∂s).
4. Not all relations are linearly decodable; some relations are reliably predicted but do not exhibit linear relational embeddings.

## Methodology
1. Extract Linear Relational Embeddings (LREs) by computing the mean Jacobian W and bias b from n=8 examples using first-order Taylor approximation: W = E[∂F/∂

In [4]:
# Read the CodeWalkthrough to understand the implementation details
with open(os.path.join(repo_path, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough_content = f.read()
print("=== CODE WALKTHROUGH ===")
print(walkthrough_content[:6000])

=== CODE WALKTHROUGH ===
# Linearity of Relation Decoding in Transformer LMs

This repository contains the code used for the experiments in the paper [Linearity of Relation Decoding in Transformer LMs](https://arxiv.org/abs/2308.09124). 

How an LM decodes a relation such as (*Miles Davis*, **plays the instrument**, *trumpet*) involves a sequence of non-linear computations spanning multiple layers. But, in this work we show that for a subset of relations this highly non-linear decoding procedure can be approximated by a simple *linear transformation* ($\textbf{LRE}$) on the subject representation $\mathbf{s}$ at some intermediate layer. 

<p align="center">
<img src="schematic-wide.png" style="width:80%;"/>
</p>

Please check [lre.baulab.info](https://lre.baulab.info/) for more information.



## Setup

All code is tested on `MacOS Ventura (>= 13.1)` and `Ubuntu 20.04` using `Python >= 3.10`. It uses a lot of newer Python features, so the Python version is a strict requirement.

To run

In [5]:
# Let's look at the evaluation directory to see existing evaluations
import os
eval_path = os.path.join(repo_path, 'evaluation')
print("=== EVALUATION DIRECTORY ===")
for item in os.listdir(eval_path):
    print(f"  {item}")

# Read the existing evaluation files
print("\n=== EXISTING EVALUATIONS ===")
for item in os.listdir(eval_path):
    if item.endswith('.json'):
        with open(os.path.join(eval_path, item), 'r') as f:
            import json
            content = json.load(f)
            print(f"\n{item}:")
            print(json.dumps(content, indent=2))

=== EVALUATION DIRECTORY ===
  self_matching.ipynb
  consistency_evaluation.json
  replications
  replication_eval

=== EXISTING EVALUATIONS ===

consistency_evaluation.json:
{
  "Checklist": {
    "CS1_Results_vs_Conclusion": "PASS",
    "CS2_Plan_vs_Implementation": "PASS"
  },
  "Rationale": {
    "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation match the results originally recorded in the code implementation notebooks. Specifically: (1) The 48% of relations achieving >60% faithfulness claim is stated in both plan.md and documentation.pdf (Section 1, Figure 3). (2) The Company CEO relation showing <6% faithfulness is explicitly mentioned in Section 4.1 of the documentation. (3) The R=0.84 correlation between faithfulness and causality is confirmed in Figure 6. (4) LRE outperforming baselines (Identity, Translation, Linear Regression) is verified in Figure 4. (5) The attribute lens recovering 54-63% R@1 on distracted prompts matches Table 3 exactly. (6) Cr

In [6]:
# Let's look at the source code to understand the key components
# First, let's examine the operators.py to understand LRE implementation
with open(os.path.join(repo_path, 'src', 'operators.py'), 'r') as f:
    operators_content = f.read()
print("=== OPERATORS.PY (first 3000 chars) ===")
print(operators_content[:3000])

=== OPERATORS.PY (first 3000 chars) ===
import itertools
import logging
import random
from dataclasses import dataclass, field
from typing import Any, Literal

from src import data, functional, models
from src.utils.typing import Layer

import baukit
import torch

logger = logging.getLogger(__name__)


@dataclass(frozen=True, kw_only=True)
class RelationOutput:
    """Predicted object tokens and their probabilities under the decoder head."""

    predictions: list[functional.PredictedToken]


@dataclass(frozen=True, kw_only=True)
class LinearRelationOutput(RelationOutput):
    """Relation output, the input `h`, and the predicted object hidden state `z`."""

    h: torch.Tensor
    z: torch.Tensor

    def as_relation_output(self) -> RelationOutput:
        return RelationOutput(predictions=self.predictions)


@dataclass(frozen=True, kw_only=True)
class RelationOperator:
    """An abstract relation operator, which maps subjects to objects."""

    def __call__(self, subject: str, **kwar

In [7]:
# Let's look at models.py to see which models are supported
with open(os.path.join(repo_path, 'src', 'models.py'), 'r') as f:
    models_content = f.read()
print("=== MODELS.PY (first 4000 chars) ===")
print(models_content[:4000])

=== MODELS.PY (first 4000 chars) ===
"""Functions for loading and interacting with pretrained language models.

This module is designed to house all the annoying branching logic
that comes with supporting analysis of many slightly different model
implementations.
"""
import argparse
import logging
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterator, Literal, Optional, Sequence, overload

from src.utils import env_utils, tokenizer_utils
from src.utils.typing import Device, Layer, Model, ModelInput, Tokenizer

import torch
import transformers

logger = logging.getLogger(__name__)

GPT_J_NAME_SHORT = "gptj"  # A useful alias for the CLI.
GPT_J_NAME = "EleutherAI/gpt-j-6B"

GPT_NEO_X_NAME_SHORT = "neox"
GPT_NEO_X_NAME = "EleutherAI/gpt-neox-20b"

LLAMA_13B_NAME = "llama-13b"
LLAMA_30B_NAME = "llama-30b"
LLAMA_NAME_SHORT = "llama"

DOWNLOADABLE_MODELS = frozenset({GPT_J_NAME, GPT_NEO_X_NAME, "gpt2-xl"})


@datacla

In [8]:
# Check what data relations are available
import json

data_path = os.path.join(repo_path, 'data')
print("=== DATA CATEGORIES ===")
for category in os.listdir(data_path):
    category_path = os.path.join(data_path, category)
    if os.path.isdir(category_path):
        print(f"\n{category}:")
        for file in os.listdir(category_path)[:5]:
            print(f"  - {file}")

# Load one example relation to understand the data structure
sample_relation = os.path.join(data_path, 'factual', 'country_capital_city.json')
with open(sample_relation, 'r') as f:
    sample_data = json.load(f)
print("\n=== SAMPLE RELATION DATA STRUCTURE ===")
print(f"Keys: {sample_data.keys()}")
print(f"Prompt templates: {sample_data.get('prompt_templates', [])[:2]}")
print(f"Number of samples: {len(sample_data.get('samples', []))}")
print(f"Sample entries (first 3): {sample_data.get('samples', [])[:3]}")

=== DATA CATEGORIES ===

commonsense:
  - work_location.json
  - task_done_by_tool.json
  - substance_phase.json
  - fruit_outside_color.json
  - fruit_inside_color.json

linguistic:
  - word_first_letter.json
  - adj_comparative.json
  - adj_antonym.json
  - word_last_letter.json
  - verb_past_tense.json

bias:
  - name_religion.json
  - characteristic_gender.json
  - occupation_age.json
  - occupation_gender.json
  - name_gender.json

factual:
  - person_occupation.json
  - presidents_birth_year.json
  - superhero_archnemesis.json
  - person_plays_position_in_sport.json
  - company_ceo.json

=== SAMPLE RELATION DATA STRUCTURE ===
Keys: dict_keys(['name', 'prompt_templates', 'prompt_templates_zs', 'properties', 'samples'])
Prompt templates: ['The capital city of {} is', 'The capital of {} is']
Number of samples: 24
Sample entries (first 3): [{'subject': 'United States', 'object': 'Washington D.C.'}, {'subject': 'Canada', 'object': 'Ottawa'}, {'subject': 'Mexico', 'object': 'Mexico Cit